# 00 · Build `manifest.csv` from the MMRDR annotation files

Notebooks 01 and 02 read `DatasetRT/manifest.csv` and `DatasetRT/metadata.csv`. This notebook writes the manifest
(with absolute image paths for your Drive) and copies the simulated descriptor file from the repository's `data/`
folder. Run it once before notebook 01. It changes no result: if an earlier manifest exists, the notebook checks that
the new one has the same rows, the same modality presence and the same labels.

**Manifest rule.**
* One row per MMRDR file identifier (`tr000001`, `ts000001`, …) and eye (`lr` 0 → `OS`, 1 → `OD`).
* CFP, OCT and UWF images with the same identifier and eye share a row. The MMRDR subsets come from different
  sources, so this pairing is by identifier only and does not imply that the images show the same eye.
* `dr_grade` = CFP grade if a CFP image is present, otherwise the UWF grade, otherwise the OCT grade (a DME grade).
* Result: 17,224 rows.

In [ ]:
from pathlib import Path
import shutil
import numpy as np, pandas as pd

from google.colab import drive
drive.mount("/content/drive")

PROJECT_ROOT = Path("/content/drive/MyDrive/MMRDR_Project")
DATA_ROOT = PROJECT_ROOT / "DatasetRT"          # contains MMRDR-CFP/, MMRDR-OCT/, MMRDR-UWF/
REPO_DATA = PROJECT_ROOT / "data"               # this repository's data/ folder (metadata.csv)

In [ ]:
MODALITIES = {"fundus": ("MMRDR-CFP", "FP.csv"), "oct": ("MMRDR-OCT", "OCT.csv"), "uwf": ("MMRDR-UWF", "UWF.csv")}

def read_modality(key):
    folder, csv_name = MODALITIES[key]
    ann = pd.read_csv(DATA_ROOT / folder / csv_name)
    out = pd.DataFrame({
        "sample_id": ann["image"].map(lambda s: Path(s).stem),
        "eye": ann["lr"].map({0: "OS", 1: "OD"}),
        f"{key}_path": ann["image"].map(lambda s: str(DATA_ROOT / folder / s)),
        f"{key}_grade": ann["grade"].astype(float),
        f"{key}_lesion": ann["lesion"] if "lesion" in ann.columns else np.nan,
    })
    assert not out.duplicated(["sample_id", "eye"]).any()
    print(f"{key:>6}: {len(out):,} images")
    return out

frames = [read_modality(k) for k in ["fundus", "oct", "uwf"]]
manifest = frames[0]
for f in frames[1:]:
    manifest = manifest.merge(f, on=["sample_id", "eye"], how="outer")
manifest["patient_id"] = manifest["sample_id"]
manifest["dr_grade"] = manifest["fundus_grade"].fillna(manifest["uwf_grade"]).fillna(manifest["oct_grade"])
manifest["dme_label"] = np.nan
manifest = manifest.sort_values(["sample_id", "eye"]).reset_index(drop=True)
manifest = manifest[["sample_id", "patient_id", "eye", "fundus_path", "oct_path", "uwf_path", "dr_grade", "dme_label",
                     "fundus_grade", "oct_grade", "uwf_grade", "fundus_lesion", "oct_lesion", "uwf_lesion"]]
print("Manifest rows:", len(manifest))
assert len(manifest) == 17224

In [ ]:
target = DATA_ROOT / "manifest.csv"
if target.exists():
    old = pd.read_csv(target).set_index(["sample_id", "eye"])
    new = manifest.set_index(["sample_id", "eye"])
    assert list(old.index) == list(new.index), "row order or keys differ from the existing manifest"
    for col in ["fundus_path", "oct_path", "uwf_path"]:
        assert (old[col].notna().values == new[col].notna().values).all(), f"presence of {col} differs"
    assert np.allclose(old["dr_grade"].values, new["dr_grade"].values)
    print("Existing manifest has the same rows, order, modality presence and labels.")
manifest.to_csv(target, index=False)
shutil.copy(REPO_DATA / "metadata.csv", DATA_ROOT / "metadata.csv")
print("Wrote", target, "and copied metadata.csv to", DATA_ROOT)